# Create INCa Awards (Institut National Du Cancer, France)

Creates INCa awards from the official funded-projects CSV published by INCa on data.gouv.fr.

**Prerequisites:** run `scripts/local/inca_to_s3.py` to download + upload first.

**Data source:** data.gouv.fr CKAN organization `institut-national-du-cancer`, dataset `projets-de-recherche-soutenus-par-linstitut-2008-2022`. Direct CSV; ~2.2K projects 2008-2022.

**S3 location:** `s3a://openalex-ingest/awards/inca/inca_projects.parquet`

**INCa funder in OpenAlex:** funder_id 4320323807 · display_name "Institut National Du Cancer" · FR.

**Input columns from inca_to_s3.py:**
- funder_award_id -> `{Funder}-{Reference}` (e.g. INCa-513, INCa-DGOS-6368) — matches cited grant format in papers (verified against OpenAIRE INCa records).
- title, description (EN abstract preferred, FR fallback), amount_eur (EUR, 0/neg already NULLed §6.7).
- pi_given_name, pi_family_name, institution_name, institution_city (city kept in raw but not surfaced to the awards schema).
- call_year, call_description (the funding programme), call_reference (programme code), acronym (1.9% — short project nickname).
- funder_label = raw `Funder` field (INCa / INCa-DGOS / INCa-Fondation ARC-LNCC / …) for co-funded grants.

**Notes:**
- All 2,212 rows have title + EUR amount + PI + institution + call_year.
- description has 100% coverage when EN/FR are merged (EN alone 79%, FR alone 43%).
- funding_type heuristic on `call_description`: training/doctoral keywords -> fellowship, else research.
- provenance `inca_data_gouv`, priority 183.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.inca_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/inca/inca_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.inca_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.inca_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.inca_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

Must return exactly 1 row. If 0, STOP — the funder is missing from `openalex.common.funder` and the CROSS JOIN below would silently emit zero rows.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320323807;

## Step 2: Create INCa Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.inca_awards
USING delta
AS
WITH
inca_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320323807  -- Institut National Du Cancer
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount_eur AS DOUBLE) > 0 THEN TRY_CAST(g.amount_eur AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount_eur AS DOUBLE) > 0 THEN 'EUR' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN LOWER(g.call_description) RLIKE '(doctor|doctorat|thèse|formation|jeune chercheur|young inves|fellow|bourse|post.?doc)' THEN 'fellowship'
            ELSE 'research'
        END as funding_type,
        g.call_description as funder_scheme,
        'inca_data_gouv' as provenance,
        CAST(CONCAT(g.call_year, '-01-01') AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.call_year AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL OR g.institution_name IS NOT NULL THEN
                struct(
                    INITCAP(g.pi_given_name) as given_name,
                    INITCAP(g.pi_family_name) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    struct(
                        g.institution_name as name,
                        'France' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation,
                    CAST(NULL AS DATE) as role_start
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >>) as investigators,
        'https://www.data.gouv.fr/datasets/projets-de-recherche-soutenus-par-linstitut-2008-2022/' as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.inca_raw g
    CROSS JOIN inca_funder f
    WHERE g.funder_award_id IS NOT NULL
      AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'inca_data_gouv' AND priority = 183;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    183 as priority
FROM openalex.awards.inca_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_inca_awards FROM openalex.awards.inca_awards;

In [ ]:
%sql
SELECT id, funder_award_id, display_name, funding_type, amount, currency, start_year, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.inca_awards LIMIT 10;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.inca_awards GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt FROM openalex.awards.inca_awards WHERE funder_scheme IS NOT NULL GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
-- §6.3 completeness query. NOTE: pct_with_amount expected ~91% (the 2008-pre records have empty Amount cells in the source; never imputed); pct_with_description expected ~100% (EN abstract preferred, FR fallback).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_date) as has_start_date,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description) * 100.0, COUNT(*)), 1) as pct_with_description
FROM openalex.awards.inca_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.inca_awards WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as grant_count
FROM openalex.awards.inca_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY lead_investigator.affiliation.name ORDER BY grant_count DESC LIMIT 20;